# B-06: spike-aware two-stage hurdle model (occurrence x magnitude, D-43)

B-05 (arcsinh target transform) was a documented negative result: squashing spikes in training trades spike
accuracy for bulk accuracy. This notebook tests the structural alternative flagged at the time: split the
problem into **(1) a classifier** predicting whether the horizon-h target will be a "spike" (> tower-specific
q90, frozen on 2018-2021 training years), **(2) two magnitude regressors** -- one trained only on non-spike
training rows, one only on spike rows -- and **(3) a soft probability blend**:

`prediction = P(spike|x) * magnitude_spike(x) + (1 - P(spike|x)) * magnitude_nonspike(x)`

This is the correct decomposition of `E[y|x]`. Unlike B-05, nothing about the target is transformed -- the
*architecture* is allowed to specialise per regime instead. Same data, features, CV, and partial-pooled
training (T2+T4+T9 + tower dummies) as B-03; same RF/XGB hyperparameters (no new HPO this round -- bounded
iteration, D-41 norm). Both tracks (hourly A, daily B) per the session's scope decision.

Reports: regression metrics (R2/RMSE/MAE/MBE/skill) for `Hurdle-RF`/`Hurdle-XGB` alongside the existing
persistence/clim/RF/XGB rows; **event-detection precision/recall/F1** for the spike classifier; and a
**spike-day vs non-spike-day conditional RMSE breakdown** against B-03 -- the actual diagnostic for whether
this structural split helps where B-05's transform didn't.

In [1]:
from pathlib import Path
import sys, datetime, numpy as np, pandas as pd
sys.path.insert(0, "../../src")
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                              precision_score, recall_score, f1_score)
from xgboost import XGBRegressor, XGBClassifier
from evaluation.metrics import full_metrics
HOURLY=Path("../../data/Hourly"); RESULTS=Path("../../results")
ALGOS=["RF","XGB"]
HORIZONS={"A":[1,6,12,24,48],"B":[1,3,7,14]}
T2_FOLDS=[("2018-06-30","2018-07-01","2018-12-31"),("2018-12-31","2019-01-01","2019-05-31")]
DUM=["is_t2","is_t4","is_t9"]

matv2=pd.read_csv(HOURLY/"forecast_features_v2.csv",low_memory=False)
matv2["Datetime"]=pd.to_datetime(matv2["Datetime"],format="mixed")
AR_A=[c for c in matv2.columns if c.startswith("ar_")]
FX_A=[c for c in matv2.columns if c.startswith("fx")]
print("hourly v2 rows",len(matv2),"| AR_A",len(AR_A),"| FX_A",len(FX_A))

hourly v2 rows 210459 | AR_A 20 | FX_A 26


## 1  Spike thresholds (per tower, frozen on 2018-2021 train years, q90)

In [2]:
dv=pd.read_csv(HOURLY/"forecast_daily_v2.csv",low_memory=False)
dv["Datetime"]=pd.to_datetime(dv["Datetime"],format="mixed")

def spike_thresholds(df, towers=(2,4,9)):
    th={}
    for t in towers:
        s=df[df.tower==t]
        tr=s[(s.Datetime.dt.year>=2018)&(s.Datetime.dt.year<=2021)]["y_observed"].dropna()
        th[t]=float(tr.quantile(0.90)) if len(tr)>=20 else np.nan
    return th

THRESH_A=spike_thresholds(matv2)
THRESH_B=spike_thresholds(dv)
print("Track A (hourly) q90 thresholds:",THRESH_A)
print("Track B (daily)  q90 thresholds:",THRESH_B)

Track A (hourly) q90 thresholds: {2: 71.457395214, 4: 78.99610909200005, 9: 109.72678194800001}
Track B (daily)  q90 thresholds: {2: 81.90970496835237, 4: 71.08691920402013, 9: 116.4771911111111}


## 2  Helpers -- classifier + spike/non-spike regressors + soft blend

In [3]:
def fit_reg(algo, tr, feat, track="A"):
    imp=SimpleImputer(strategy="mean"); Xi=imp.fit_transform(tr[feat].values)
    if algo=="RF":
        p=dict(min_samples_leaf=5) if track=="A" else dict(min_samples_leaf=10,max_features=0.5)
        m=RandomForestRegressor(n_estimators=500,n_jobs=-1,random_state=42,**p)
    else:
        p=dict(max_depth=6,learning_rate=0.05,n_estimators=500) if track=="A" \
          else dict(max_depth=2,learning_rate=0.02,n_estimators=400,min_child_weight=10)
        m=XGBRegressor(subsample=0.8,colsample_bytree=0.8,n_jobs=-1,random_state=42,**p)
    m.fit(Xi,tr["target"].values); return m,imp

def fit_clf(algo, tr, feat, track="A"):
    imp=SimpleImputer(strategy="mean"); Xi=imp.fit_transform(tr[feat].values)
    y=tr["spike"].values.astype(int)
    if y.sum()<5 or y.sum()>len(y)-5:
        return None,imp  # degenerate: not enough of one class
    if algo=="RF":
        p=dict(min_samples_leaf=5) if track=="A" else dict(min_samples_leaf=10,max_features=0.5)
        m=RandomForestClassifier(n_estimators=500,n_jobs=-1,random_state=42,class_weight="balanced",**p)
    else:
        p=dict(max_depth=6,learning_rate=0.05,n_estimators=500) if track=="A" \
          else dict(max_depth=2,learning_rate=0.02,n_estimators=400,min_child_weight=10)
        spw=float((y==0).sum())/max(1,(y==1).sum())
        m=XGBClassifier(subsample=0.8,colsample_bytree=0.8,n_jobs=-1,random_state=42,
                         scale_pos_weight=spw,eval_metric="logloss",**p)
    m.fit(Xi,y); return m,imp

def predict_hurdle(clf,clf_imp,reg_spike,reg_spike_imp,reg_base,reg_base_imp,feat,X):
    p_spike = clf.predict_proba(clf_imp.transform(X[feat].values))[:,1] if clf is not None else np.zeros(len(X))
    yhat_spike = reg_spike.predict(reg_spike_imp.transform(X[feat].values))
    yhat_base  = reg_base.predict(reg_base_imp.transform(X[feat].values))
    return p_spike*yhat_spike + (1-p_spike)*yhat_base, p_spike

def rmse(y,p): return float(np.sqrt(mean_squared_error(y,p)))
def metrics(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    r2=r2_score(y,p) if (len(y)>1 and np.var(y)>0) else np.nan
    return rmse(y,p), float(mean_absolute_error(y,p)), r2, float(np.mean(p-y))

def climatology(tr, te, unit):
    t=te["tower"].iloc[0]; trt=tr[tr.tower==t]
    if len(trt)<24: trt=tr
    gl=float(trt["target"].mean())
    if unit=="h":
        g=trt.assign(mo=trt.ttime.dt.month,hr=trt.ttime.dt.hour).groupby(["mo","hr"])["target"].mean()
        keys=list(zip(te.ttime.dt.month,te.ttime.dt.hour)); return np.array([g.get(k,gl) for k in keys])
    g=trt.assign(mo=trt.ttime.dt.month).groupby("mo")["target"].mean()
    return np.array([g.get(m,gl) for m in te.ttime.dt.month])

def build_hourly_tables():
    T={t: matv2[matv2.tower==t].set_index("Datetime").sort_index() for t in [2,4,9]}
    return T, AR_A, FX_A

def build_daily_tables():
    AR_B=[c for c in dv.columns if c.startswith("ar_")]; FX_B=[c for c in dv.columns if c.startswith("fx")]
    T={t: dv[dv.tower==t].set_index("Datetime").sort_index() for t in [2,4,9]}
    return T, AR_B, FX_B

def aligned(T, h, unit, ar_cols, fx_cols, thresh):
    parts=[]; off=pd.Timedelta(hours=h) if unit=="h" else pd.Timedelta(days=h)
    for t,df in T.items():
        f=df[ar_cols+DUM].copy()
        for c in fx_cols: f[c]=df[c].shift(-h)
        f["target"]=df["y_observed"].shift(-h); f["persist"]=df["y_gapfilled"]
        f["tower"]=t; f["ttime"]=df.index+off
        f["spike"]=(f["target"]>thresh.get(t,np.inf)).astype(float)
        f.loc[f["target"].isna(),"spike"]=np.nan
        parts.append(f)
    return pd.concat(parts)

## 3  Run one horizon (towers 4/9 split + Tower 2 expanding folds)

In [4]:
def emit(track,h,t,bydict,rp,rc,y_naive,event=None):
    rows=[]
    for model,(y,p) in bydict.items():
        r,mae,r2,mbe=metrics(y,p)
        fm=full_metrics(y,p,y_naive=y_naive)
        row=dict(track=track,horizon=h,tower=t,model=model,RMSE=round(r,3),MAE=round(mae,3),
            R2=(round(r2,3) if np.isfinite(r2) else np.nan),MBE=round(mbe,3),n_test=len(y),
            skill_persist=round(1-r/rp,3) if rp>0 else np.nan,
            skill_clim=round(1-r/rc,3) if rc>0 else np.nan,
            WAPE=round(fm["WAPE"],4) if np.isfinite(fm["WAPE"]) else np.nan,
            MASE=round(fm["MASE"],4) if np.isfinite(fm["MASE"]) else np.nan,
            sMAPE=round(fm["sMAPE"],4) if np.isfinite(fm["sMAPE"]) else np.nan,
            MAPE=round(fm["MAPE"],4) if np.isfinite(fm["MAPE"]) else np.nan,
            MAPE_n_excluded=fm["MAPE_n_excluded"],
            precision=np.nan,recall=np.nan,f1=np.nan,
            rmse_spike=np.nan,rmse_nonspike=np.nan,n_spike_test=np.nan)
        if event is not None and model.startswith("Hurdle"):
            row.update(event)
        rows.append(row)
    return rows

def run_horizon(track, h, unit, ar_cols, fx_cols, T, thresh):
    feat=ar_cols+fx_cols+DUM; A=aligned(T,h,unit,ar_cols,fx_cols,thresh); rows=[]
    tr=A[A.ttime.dt.year.between(2018,2021) & A.target.notna()]
    tr_base=tr[tr.spike==0]; tr_spike=tr[tr.spike==1]
    fitted_reg={a:fit_reg(a,tr,feat,track) for a in ALGOS}
    fitted_base={a:fit_reg(a,tr_base,feat,track) for a in ALGOS}
    fitted_spike={a:fit_reg(a,tr_spike,feat,track) for a in ALGOS}
    fitted_clf={a:fit_clf(a,tr,feat,track) for a in ALGOS}
    for t in [4,9]:
        te=A[(A.tower==t)&A.ttime.dt.year.isin([2022,2023])&A.target.notna()]
        if len(te)<10: continue
        bd={"persist":(te["target"].values,te["persist"].values),
            "clim":(te["target"].values,climatology(tr,te,unit))}
        for a in ALGOS:
            m,imp=fitted_reg[a]; bd[a]=(te["target"].values,m.predict(imp.transform(te[feat].values)))
        rp=rmse(*bd["persist"]); rc=rmse(*bd["clim"]); y_naive=bd["persist"][1]
        for a in ALGOS:
            clf,clf_imp=fitted_clf[a]; rb,rb_imp=fitted_base[a]; rs,rs_imp=fitted_spike[a]
            yhat,p_spike=predict_hurdle(clf,clf_imp,rs,rs_imp,rb,rb_imp,feat,te)
            bd[f"Hurdle-{a}"]=(te["target"].values,yhat)
            y_true_cls=te["spike"].values
            event=dict(n_spike_test=int(np.nansum(y_true_cls)))
            if clf is not None and np.nansum(y_true_cls)>0 and np.nansum(1-y_true_cls)>0:
                pred_cls=(p_spike>0.5).astype(int)
                event["precision"]=round(float(precision_score(y_true_cls,pred_cls,zero_division=0)),3)
                event["recall"]=round(float(recall_score(y_true_cls,pred_cls,zero_division=0)),3)
                event["f1"]=round(float(f1_score(y_true_cls,pred_cls,zero_division=0)),3)
            spk_mask=y_true_cls==1; nsp_mask=y_true_cls==0
            if spk_mask.sum()>0: event["rmse_spike"]=round(rmse(te["target"].values[spk_mask],yhat[spk_mask]),3)
            if nsp_mask.sum()>0: event["rmse_nonspike"]=round(rmse(te["target"].values[nsp_mask],yhat[nsp_mask]),3)
            rows+=emit(track,h,t,{f"Hurdle-{a}":bd[f"Hurdle-{a}"]},rp,rc,y_naive,event=event)
        rows+=emit(track,h,t,{k:v for k,v in bd.items() if not k.startswith("Hurdle")},rp,rc,y_naive)
    acc={k:([],[]) for k in ["persist","clim"]+ALGOS+[f"Hurdle-{a}" for a in ALGOS]}
    for cut,s,e in T2_FOLDS:
        trf=A[(A.ttime<=pd.Timestamp(cut)) & A.target.notna()]
        trf_base=trf[trf.spike==0]; trf_spike=trf[trf.spike==1]
        te=A[(A.tower==2)&(A.ttime>=pd.Timestamp(s))&(A.ttime<=pd.Timestamp(e))&A.target.notna()]
        if len(te)<5 or len(trf)<50: continue
        y=te["target"].values
        acc["persist"][0].append(y); acc["persist"][1].append(te["persist"].values)
        acc["clim"][0].append(y); acc["clim"][1].append(climatology(trf,te,unit))
        for a in ALGOS:
            m,imp=fit_reg(a,trf,feat,track); acc[a][0].append(y); acc[a][1].append(m.predict(imp.transform(te[feat].values)))
            clf,clf_imp=fit_clf(a,trf,feat,track)
            rb,rb_imp=fit_reg(a,trf_base,feat,track) if len(trf_base)>=10 else fit_reg(a,trf,feat,track)
            rs,rs_imp=fit_reg(a,trf_spike,feat,track) if len(trf_spike)>=10 else fit_reg(a,trf,feat,track)
            yhat,_=predict_hurdle(clf,clf_imp,rs,rs_imp,rb,rb_imp,feat,te)
            acc[f"Hurdle-{a}"][0].append(y); acc[f"Hurdle-{a}"][1].append(yhat)
    if acc["persist"][0]:
        bd={k:(np.concatenate(v[0]),np.concatenate(v[1])) for k,v in acc.items()}
        rp=rmse(*bd["persist"]); rc=rmse(*bd["clim"])
        rows+=emit(track,h,2,bd,rp,rc,bd["persist"][1])
    return rows

## 4  Run both tracks, all horizons

In [5]:
TA,AR_A_,FX_A_=build_hourly_tables(); TB,AR_B_,FX_B_=build_daily_tables()
ALL=[]
for h in HORIZONS["A"]:
    ALL+=run_horizon("A",h,"h",AR_A_,FX_A_,TA,THRESH_A); print("Track A h=",h,"done",flush=True)
for h in HORIZONS["B"]:
    ALL+=run_horizon("B",h,"D",AR_B_,FX_B_,TB,THRESH_B); print("Track B d=",h,"done",flush=True)
R=pd.DataFrame(ALL); print("rows",len(R))

Track A h= 1 done


Track A h= 6 done


Track A h= 12 done


Track A h= 24 done


Track A h= 48 done


Track B d= 1 done


Track B d= 3 done


Track B d= 7 done


Track B d= 14 done


rows 162


## 5  Results -- regression (R2) + event-detection (P/R/F1) + conditional RMSE

In [6]:
pd.set_option("display.width",220)
print("=== Track B (daily) -- R2, towers 4/9: persist/clim/RF/XGB/Hurdle-RF/Hurdle-XGB ===")
print(R[(R.track=="B")&(R.tower.isin([4,9]))]
      .pivot_table(index=["tower","model"],columns="horizon",values="R2").round(3).to_string())
print("\n=== Track A (hourly) -- R2, towers 4/9 ===")
print(R[(R.track=="A")&(R.tower.isin([4,9]))]
      .pivot_table(index=["tower","model"],columns="horizon",values="R2").round(3).to_string())
print("\n=== Event-detection (Hurdle only) -- precision/recall/F1, towers 4/9 ===")
ev=R[(R.tower.isin([4,9]))&(R.model.str.startswith("Hurdle"))]
print(ev.pivot_table(index=["track","tower","model"],columns="horizon",values="f1").round(3).to_string())
print("\n=== Conditional RMSE (Hurdle) -- spike-day vs non-spike-day, towers 4/9 ===")
print(ev[["track","tower","model","horizon","rmse_spike","rmse_nonspike","n_spike_test","n_test"]].sort_values(["track","tower","model","horizon"]).to_string(index=False))
R.to_csv(RESULTS/"b06_summary.csv",index=False); print("\nsaved results/b06_summary.csv")

=== Track B (daily) -- R2, towers 4/9: persist/clim/RF/XGB/Hurdle-RF/Hurdle-XGB ===
horizon              1      3      7      14
tower model                                 
4     Hurdle-RF   0.253  0.206  0.174  0.154
      Hurdle-XGB  0.179  0.026  0.003 -0.078
      RF          0.357  0.345  0.287  0.270
      XGB         0.362  0.313  0.284  0.259
      clim       -0.022 -0.022 -0.022 -0.022
      persist     0.440  0.106 -0.260 -1.163
9     Hurdle-RF   0.130  0.098  0.093  0.059
      Hurdle-XGB -0.280 -0.395 -0.332 -0.431
      RF          0.388  0.350  0.364  0.342
      XGB         0.324  0.271  0.303  0.275
      clim        0.045  0.045  0.045  0.045
      persist     0.231 -0.234 -0.726 -0.683

=== Track A (hourly) -- R2, towers 4/9 ===
horizon              1      6      12     24     48
tower model                                        
4     Hurdle-RF   0.142  0.089  0.109  0.110  0.095
      Hurdle-XGB  0.140  0.058  0.076  0.078  0.064
      RF          0.136  0.044  0.

## 6  Compare Hurdle vs B-03 (same-model RF/XGB rows + conditional RMSE benchmark)

In [7]:
b03=pd.read_csv(RESULTS/"b03_summary.csv")
key=["track","tower","horizon"]
# B-03's single-regressor RMSE, split into spike/non-spike test days using THIS notebook's same thresholds
def b03_conditional_rmse(track, h, t, thresh, T, ar_cols, fx_cols, algo="RF"):
    feat=ar_cols+fx_cols+DUM
    A=aligned(T,h,"h" if track=="A" else "D",ar_cols,fx_cols,thresh)
    tr=A[A.ttime.dt.year.between(2018,2021) & A.target.notna()]
    te=A[(A.tower==t)&A.ttime.dt.year.isin([2022,2023])&A.target.notna()]
    if len(te)<10: return np.nan,np.nan
    m,imp=fit_reg(algo,tr,feat,track)
    p=m.predict(imp.transform(te[feat].values)); y=te["target"].values; spk=te["spike"].values==1
    rs=rmse(y[spk],p[spk]) if spk.sum()>0 else np.nan
    rn=rmse(y[~spk],p[~spk]) if (~spk).sum()>0 else np.nan
    return round(rs,3), round(rn,3)

rows=[]
for track,h_list,T,ar,fx,thresh in [("A",HORIZONS["A"],TA,AR_A_,FX_A_,THRESH_A),("B",HORIZONS["B"],TB,AR_B_,FX_B_,THRESH_B)]:
    for h in h_list:
        for t in [4,9]:
            for algo in ALGOS:
                rs,rn=b03_conditional_rmse(track,h,t,thresh,T,ar,fx,algo)
                rows.append(dict(track=track,horizon=h,tower=t,model=algo,rmse_spike_b03=rs,rmse_nonspike_b03=rn))
B03COND=pd.DataFrame(rows)
comp=ev.merge(B03COND.rename(columns={"model":"base_model"}),
              left_on=["track","horizon","tower"],right_on=["track","horizon","tower"])
comp=comp[comp.apply(lambda r: r["model"]==f"Hurdle-{r['base_model']}",axis=1)]
comp["d_rmse_spike"]=comp["rmse_spike"]-comp["rmse_spike_b03"]
comp["d_rmse_nonspike"]=comp["rmse_nonspike"]-comp["rmse_nonspike_b03"]
print("=== Hurdle vs B03-equivalent single regressor: delta conditional RMSE (negative = Hurdle better) ===")
print(comp[(comp.tower.isin([4,9]))][["track","tower","base_model","horizon","rmse_spike_b03","rmse_spike","d_rmse_spike","rmse_nonspike_b03","rmse_nonspike","d_rmse_nonspike"]]
      .sort_values(["track","tower","base_model","horizon"]).to_string(index=False))
print("\n=== Mean delta conditional RMSE by track/tower/algo (towers 4/9) ===")
print(comp[comp.tower.isin([4,9])].groupby(["track","tower","base_model"])[["d_rmse_spike","d_rmse_nonspike"]].mean().round(3).to_string())

=== Hurdle vs B03-equivalent single regressor: delta conditional RMSE (negative = Hurdle better) ===
track  tower base_model  horizon  rmse_spike_b03  rmse_spike  d_rmse_spike  rmse_nonspike_b03  rmse_nonspike  d_rmse_nonspike
    A      4         RF        1         372.065     363.006        -9.059             65.183         69.900            4.717
    A      4         RF        6         384.219     378.769        -5.450             73.081         69.027           -4.054
    A      4         RF       12         379.878     376.346        -3.532             70.154         67.274           -2.880
    A      4         RF       24         380.996     372.853        -8.143             74.233         69.268           -4.965
    A      4         RF       48         386.334     378.036        -8.298             73.766         68.471           -5.295
    A      4        XGB        1         371.889     361.565       -10.324             68.373         71.112            2.739
    A      4     

## 7  Append to benchmarks.csv (B06)

In [8]:
bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="B06"]
rows=[]
for _,r in R.iterrows():
    rows.append({"replication":"B06","model":r["model"],"tower":f"Tower {int(r.tower)}",
        "feature_set":"enriched_v2 + spike-aware two-stage hurdle (occurrence x magnitude, soft blend)",
        "track":r["track"],"horizon":int(r["horizon"]),
        "split":"fc_main" if r.tower in (4,9) else "fc_t2folds",
        "R2":r["R2"],"RMSE":r["RMSE"],"MAE":r["MAE"],"MBE":r["MBE"],"n_test":int(r["n_test"]),
        "skill_persist":r["skill_persist"],"skill_clim":r["skill_clim"],
        "WAPE":r["WAPE"],"MASE":r["MASE"],"sMAPE":r["sMAPE"],"MAPE":r["MAPE"],"MAPE_n_excluded":r["MAPE_n_excluded"],
        "precision":r["precision"],"recall":r["recall"],"f1":r["f1"],
        "date":today,"notes":"B06 spike-aware two-stage hurdle (q90 threshold, soft P(spike) blend, no new HPO); compare vs B03 single regressor (D-43); WAPE/MASE/sMAPE/MAPE added D-44b"})
new=pd.DataFrame(rows); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
print(f"Wrote {len(new)} B06 rows. Total {len(comb)}.")

Wrote 162 B06 rows. Total 3665.
